# ETL Pipeline for Macroeconomic Data: World Bank GDP (2010–2024)
**Author:** Oyinlayefa Mezeh  
**Date:** March 2026  

**Objective:** This notebook ingests raw World Bank macroeconomic data (GDP in current USD) and transforms it from a "Wide" matrix into a "Long" longitudinal format. This creates the final downstream "Impact" variable for our causal chain, reflecting the macroeconomic degradation resulting from the cumulative shocks of drought, food price spikes, and civil unrest in Nigeria.

**Output:** `nigeria_gdp_master.csv`

## 1. Environment Setup & Data Extraction
The World Bank CSV files contain 4 rows of metadata before the actual headers. We bypass these using the `skiprows` parameter and immediately filter the global dataset for Nigeria (`NGA`).

In [1]:
import pandas as pd

# Load the raw World Bank data (skipping the first 4 metadata rows)
file_name = 'API_NY.GDP.PCAP.PP.CD_DS2_en_csv_v2_35.csv'
df_wb = pd.read_csv(file_name, skiprows=4)

# Filter for Nigeria using the ISO3 Country Code
df_nga = df_wb[df_wb['Country Code'] == 'NGA'].copy()

print(f"✅ Raw Data Loaded and filtered for {df_nga['Country Name'].iloc[0]}.")

✅ Raw Data Loaded and filtered for Nigeria.


## 2. Transformation: Reshaping Wide to Long (Melting)
To align this dataset with our ACLED, WFP, and EM-DAT datasets, we must pivot the years (currently spread across multiple columns) into a single `year` column and a single `gdp_current_usd` value column for the 2010–2024 scope.

In [2]:
# Define the academic timeline
target_years = [str(y) for y in range(2010, 2025)]

# Ensure we only try to extract years that actually exist in the CSV
available_years = [y for y in target_years if y in df_nga.columns]

# Subset the dataframe to just the Country Code and our target years
cols_to_keep = ['Country Code'] + available_years
df_subset = df_nga[cols_to_keep]

# Perform the Pandas 'Melt' to pivot the data
df_melted = df_subset.melt(
    id_vars=['Country Code'],
    var_name='year',
    value_name='gdp_per_capita_usd'
)

# Clean up column names and data types for the join
df_melted = df_melted.rename(columns={'Country Code': 'iso3'})
df_melted['year'] = df_melted['year'].astype(int)

# Sort chronologically
df_melted = df_melted.sort_values('year').reset_index(drop=True)

print(f"✅ Data reshaped: Extracted {len(df_melted)} years of continuous GDP records.")

✅ Data reshaped: Extracted 15 years of continuous GDP records.


## 3. Data Audit & Export
The macroeconomic index is now normalized. It will serve as the final national-level impact variable in the "Grand Merge" phase.

In [3]:
# Export the master file
output_wb = 'nigeria_gdp_master.csv'
df_melted.to_csv(output_wb, index=False)

print(f"💾 GDP Master Dataset successfully saved to: {output_wb}")

# Display a preview to verify the transformation
display(df_melted.head())

💾 GDP Master Dataset successfully saved to: nigeria_gdp_master.csv


,iso3,year,gdp_per_capita_usd
0,NGA,2010,6301.183725
1,NGA,2011,6585.361933
2,NGA,2012,6670.838262
3,NGA,2013,7004.131196
4,NGA,2014,7396.128355
